# 📊 NER vs SOMEF — Evaluation on GitHub README Files (Enhanced)

This notebook evaluates your **trained NER model** against **SOMEF** on unseen README texts.

### What you get
- ⚙️ Load saved NER model (from `FINAL_DIR`)
- 🧠 Run your model vs **SOMEF** on the same evaluation set
- 🧮 Compute robust NER metrics (overall + per-label)
- 📈 Visualize results (per-label bars & confusion matrices)
- 🧪 Inspect mismatches easily
- 🗂️ **Export metrics** to CSV and JSON for reporting

> Assumes your eval dataset follows the same structure as training (Label Studio-like).


## 1) 📦 Install Dependencies

In [7]:
!pip install -U transformers datasets evaluate seqeval torch tqdm matplotlib scikit-learn somef --quiet
import sys
print("Python:", sys.version)

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
        Using cached bs4-0.0.1.tar.gz (1.1 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
        Using cached Click-7.0-py2.py3-none-any.whl.metadata (3.5 kB)
        Using cached click_option_group-0.5.3-py3-none-any.whl.metadata (7.2 kB)
        Using cached Markdown-3.3.6-py3-none-any.whl.metadata (4.6 kB)
        Using cached matplotlib-3.5.0.tar.gz (35.0 MB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to buil

## 2) 🧰 Imports & Setup

In [6]:
import os, json, logging, re, tempfile
from typing import List, Dict, Any

import numpy as np
import torch
from tqdm.auto import tqdm
import evaluate
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification

# SOMEF check
try:
    from somef.cli import cli_main as somef_cli_main
    SOMEF_AVAILABLE = True
except Exception as e:
    SOMEF_AVAILABLE = False

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("eval_pipeline")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {DEVICE}")

ModuleNotFoundError: No module named 'numpy'

## 3) ⚙️ Configuration

In [ ]:
EVAL_DATA_PATH = "/mnt/data/eval_dataset.json"
FINAL_DIR      = "./ner_lora_model_final"

MAX_LENGTH = 512
THRESHOLD  = 0.70
MAX_GAP    = 1

EXPORT_DIR = "./eval_exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

logger.info("Config loaded.")

## 4) 📂 Load Evaluation Dataset

In [ ]:
assert os.path.exists(EVAL_DATA_PATH), f"Eval dataset not found: {EVAL_DATA_PATH}"
with open(EVAL_DATA_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)
logger.info(f"Loaded {len(eval_data)} evaluation samples.")

def parse_eval_rows(items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows = []
    for item in items:
        text = item.get("data", {}).get("text", "")
        if not text:
            continue
        entities = []
        for ann in item.get("annotations", []):
            for r in ann.get("result", []):
                v = r.get("value", {})
                start, end = v.get("start"), v.get("end")
                labels = v.get("labels", [])
                if start is None or end is None or not labels:
                    continue
                for lbl in labels:
                    entities.append({"start": start, "end": end, "label": lbl})
        rows.append({"text": text, "entities": entities})
    return rows

eval_rows = parse_eval_rows(eval_data)
logger.info(f"Parsed {len(eval_rows)} rows for evaluation. "
            f"{sum(1 for r in eval_rows if r['entities'])} rows with entities.")

## 5) 🧠 Load Trained NER Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
model     = AutoModelForTokenClassification.from_pretrained(FINAL_DIR).to(DEVICE)
id2label  = model.config.id2label
label2id  = model.config.label2id
logger.info(f"Loaded model with {model.config.num_labels} labels.")

## 6) ⚡ NER Model Inference

In [ ]:
@torch.no_grad()
def ner_inference(text: str, threshold: float = THRESHOLD, max_gap: int = MAX_GAP):
    enc = tokenizer(text, return_tensors="pt", truncation=True, return_offsets_mapping=True, max_length=MAX_LENGTH)
    offsets = enc.pop("offset_mapping").tolist()
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    outputs = model(**enc)
    probs = torch.softmax(outputs.logits, dim=2)[0]
    confs, preds = torch.max(probs, dim=1)
    confs = confs.cpu().tolist()
    preds = preds.cpu().tolist()

    for i, c in enumerate(confs):
        if c < threshold:
            preds[i] = label2id.get("O", preds[i])

    tokens = enc["input_ids"][0].cpu().tolist()
    entities = []
    current = None
    current_confs = []

    for tok_id, pred_id, (start, end), conf in zip(tokens, preds, offsets[0], confs):
        tok = tokenizer.convert_ids_to_tokens(tok_id)
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        label = id2label[pred_id]
        clean = tok.replace("##", "")

        if label == "O":
            if current:
                current["confidence"] = float(sum(current_confs) / len(current_confs))
                entities.append(current)
                current, current_confs = None, []
            continue

        if current and current["label"] == label and (start - current["end"]) <= max_gap:
            if tok.startswith("##"):
                current["text"] += clean
            else:
                current["text"] += " " + clean
            current["end"] = end
            current_confs.append(conf)
        else:
            if current:
                current["confidence"] = float(sum(current_confs) / len(current_confs))
                entities.append(current)
            current = {"text": clean, "label": label, "start": start, "end": end}
            current_confs = [conf]

    if current:
        current["confidence"] = float(sum(current_confs) / len(current_confs))
        entities.append(current)
    return entities

logger.info("Running model inference over eval set...")
model_preds = [ner_inference(r["text"]) for r in tqdm(eval_rows, desc="Model inference")]
logger.info("Model predictions done.")

## 7) 🧠 SOMEF Extraction (mapping + caching)

In [ ]:
SOMEF_FIELD_MAPPING = {
    "license": "LICENSE",
    "license_url": "LICENSE_URL",
    "operating_system": "OPERATING_SYSTEM",
    "runtime_platform": "RUNTIME_PLATFORM",
    "software_requirements": "SOFTWARE_REQUIREMENTS",
    "citation": "CITATION",
    "reference_publication": "REFERENCE_PUBLICATION",
    "reference_publication_url": "REFERENCE_PUBLICATION_URL",
    "build_instructions": "BUILD_INSTRUCTIONS",
}

def run_somef_on_text(text: str):
    if not SOMEF_AVAILABLE:
        return {}
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".md")
    try:
        with open(tmp.name, "w", encoding="utf-8") as f:
            f.write(text)
        try:
            res = somef_cli_main(["describe", tmp.name, "--format", "json"])
        except SystemExit:
            res = None
        if not res or isinstance(res, int):
            return {}
        return res if isinstance(res, dict) else {}
    finally:
        try:
            os.unlink(tmp.name)
        except Exception:
            pass

def somef_to_spans(text: str, js: dict):
    spans = []
    if not js:
        return spans
    for k, v in js.items():
        mapped = SOMEF_FIELD_MAPPING.get(k.lower())
        if not mapped:
            continue
        if isinstance(v, list):
            for item in v:
                if isinstance(item, str):
                    idx = text.find(item)
                    if idx != -1:
                        spans.append({"start": idx, "end": idx + len(item), "label": mapped})
    return spans

_somef_cache = {}

def get_somef_spans(text: str):
    if text in _somef_cache:
        return _somef_cache[text]
    raw = run_somef_on_text(text)
    spans = somef_to_spans(text, raw)
    _somef_cache[text] = spans
    return spans

logger.info(f"SOMEF available: {SOMEF_AVAILABLE}")
logger.info("Running SOMEF over eval set...")
somef_preds = [get_somef_spans(r["text"]) for r in tqdm(eval_rows, desc="SOMEF extraction")]
logger.info("SOMEF predictions done.")

## 8) 🧮 Prepare BIO Labels for Metrics

In [ ]:
seqeval_metric = evaluate.load("seqeval")

def spans_to_bio(text: str, spans: List[Dict[str, Any]]):
    enc = tokenizer(text, truncation=True, max_length=MAX_LENGTH, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    labels = ["O"] * len(enc["input_ids"])
    for s in spans:
        start, end, lab = s["start"], s["end"], s["label"]
        first = True
        for i, (os_, oe_) in enumerate(offsets):
            if os_ == 0 and oe_ == 0:
                continue
            if os_ < end and oe_ > start:
                labels[i] = f"B-{lab}" if first else f"I-{lab}"
                first = False
    # strip CLS/SEP if present
    if len(labels) >= 2:
        labels = labels[1:-1]
    return labels

true_labels, model_labels, somef_labels = [], [], []
for row, mspans, sspans in zip(eval_rows, model_preds, somef_preds):
    true_labels.append(spans_to_bio(row["text"], row["entities"]))
    model_labels.append(spans_to_bio(row["text"], mspans))
    somef_labels.append(spans_to_bio(row["text"], sspans))

logger.info("BIO labels prepared.")

## 9) 📈 Compute Metrics — Model vs SOMEF

In [ ]:
def overall_block(title, res):
    print(f"\n=== {title} (Overall) ===")
    print(f"Precision: {res.get('overall_precision', 0):.4f} | "
          f"Recall: {res.get('overall_recall', 0):.4f} | "
          f"F1: {res.get('overall_f1', 0):.4f} | "
          f"Accuracy: {res.get('overall_accuracy', 0):.4f}")

model_results = seqeval_metric.compute(predictions=model_labels, references=true_labels)
somef_results = seqeval_metric.compute(predictions=somef_labels, references=true_labels)

overall_block("Model", model_results)
overall_block("SOMEF", somef_results)

def per_label_dict(res):
    return {k: v for k, v in res.items() if isinstance(v, dict)}

per_label_model = per_label_dict(model_results)
per_label_somef = per_label_dict(somef_results)

print("\nSample per-label (Model):")
for i, (k, v) in enumerate(list(per_label_model.items())[:10]):
    print(k, v)

## 10) 📊 Per-Label F1 — Model vs SOMEF

In [ ]:
labels = sorted(set(list(per_label_model.keys()) + list(per_label_somef.keys())))
mf1 = [per_label_model.get(l, {}).get("f1", 0.0) for l in labels]
sf1 = [per_label_somef.get(l, {}).get("f1", 0.0) for l in labels]

plt.figure(figsize=(12,6))
x = np.arange(len(labels))
plt.bar(x - 0.2, mf1, width=0.4, label="Model")
plt.bar(x + 0.2, sf1, width=0.4, label="SOMEF")
plt.xticks(x, labels, rotation=45, ha="right")
plt.ylabel("F1 score")
plt.title("Per-label F1 comparison: Model vs SOMEF")
plt.legend()
plt.tight_layout()
plt.show()

## 11) 🧭 Confusion Matrices (Token-level BIO)

In [ ]:
flat_true  = [l for seq in true_labels  for l in seq]
flat_model = [l for seq in model_labels for l in seq]
flat_somef = [l for seq in somef_labels for l in seq]

axis_labels = sorted(set(flat_true + flat_model + flat_somef))

cm_model = confusion_matrix(flat_true, flat_model, labels=axis_labels)
cm_somef = confusion_matrix(flat_true, flat_somef, labels=axis_labels)

fig, ax = plt.subplots(1, 2, figsize=(18, 6))

disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_model, display_labels=axis_labels)
disp1.plot(ax=ax[0], values_format="d", cmap="Blues", xticks_rotation=45, colorbar=False)
ax[0].set_title("Model Confusion Matrix")

disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_somef, display_labels=axis_labels)
disp2.plot(ax=ax[1], values_format="d", cmap="Greens", xticks_rotation=45, colorbar=False)
ax[1].set_title("SOMEF Confusion Matrix")

plt.tight_layout()
plt.show()

## 12) 🧪 Error Inspection

In [ ]:
def inspect_error(idx: int):
    print("\n==================== INDEX", idx, "====================\n")
    text = eval_rows[idx]["text"]
    print("TEXT (first 600 chars):\n", text[:600], "..." if len(text) > 600 else "")
    print("\nGOLD:",  eval_rows[idx]["entities"])
    print("MODEL:", model_preds[idx])
    print("SOMEF:", somef_preds[idx])

# Example:
# inspect_error(0)

## 13) 🗂️ Export Metrics (CSV & JSON)

In [ ]:
import csv

def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

overall = {
    "model": {
        "precision": model_results.get("overall_precision", 0.0),
        "recall":    model_results.get("overall_recall", 0.0),
        "f1":        model_results.get("overall_f1", 0.0),
        "accuracy":  model_results.get("overall_accuracy", 0.0),
    },
    "somef": {
        "precision": somef_results.get("overall_precision", 0.0),
        "recall":    somef_results.get("overall_recall", 0.0),
        "f1":        somef_results.get("overall_f1", 0.0),
        "accuracy":  somef_results.get("overall_accuracy", 0.0),
    }
}

per_label_model = {k: v for k, v in model_results.items() if isinstance(v, dict)}
per_label_somef = {k: v for k, v in somef_results.items() if isinstance(v, dict)}

write_json(os.path.join(EXPORT_DIR, "overall_metrics.json"), overall)
write_json(os.path.join(EXPORT_DIR, "per_label_model.json"), per_label_model)
write_json(os.path.join(EXPORT_DIR, "per_label_somef.json"), per_label_somef)

csv_path = os.path.join(EXPORT_DIR, "per_label_comparison.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["label", "model_precision", "model_recall", "model_f1", "somef_precision", "somef_recall", "somef_f1"])
    all_labs = sorted(set(list(per_label_model.keys()) + list(per_label_somef.keys())))
    for lab in all_labs:
        mp = per_label_model.get(lab, {}).get("precision", 0.0)
        mr = per_label_model.get(lab, {}).get("recall", 0.0)
        mf = per_label_model.get(lab, {}).get("f1", 0.0)
        sp = per_label_somef.get(lab, {}).get("precision", 0.0)
        sr = per_label_somef.get(lab, {}).get("recall", 0.0)
        sf = per_label_somef.get(lab, {}).get("f1", 0.0)
        w.writerow([lab, mp, mr, mf, sp, sr, sf])

print("Exports written to:", EXPORT_DIR)
print("- overall_metrics.json")
print("- per_label_model.json")
print("- per_label_somef.json")
print("- per_label_comparison.csv")

## 14) 🏁 Summary

In [ ]:
print("=== MODEL Overall ===")
print(json.dumps(overall["model"], indent=2))

print("\n=== SOMEF Overall ===")
print(json.dumps(overall["somef"], indent=2))

print("\nSaved exports to:", EXPORT_DIR)